# Multi-Agent Complaints Analysis Framework

This notebook implements a multi-agent system for analyzing consumer complaints. Each agent is specialized for a specific task in the pipeline:

1. **DataLoaderAgent**: Fetches data from CFPB API or loads from CSV and preprocesses complaint data
2. **EmbeddingAgent**: Generates text embeddings using transformer models
3. **ClusteringAgent**: Performs optimal clustering on embeddings
4. **NamingAgent**: Generates human-readable cluster names
5. **VisualizationAgent**: Creates UMAP visualizations
6. **CoordinatorAgent**: Orchestrates the entire pipeline

## Data Sources

The DataLoaderAgent supports two modes:
- **API Mode**: Fetches complaints directly from the Consumer Financial Protection Bureau (CFPB) API
- **CSV Mode**: Loads complaints from a local CSV file

API mode is recommended as it ensures you always have the latest complaint data.

In [ ]:
import pandas as pd
import numpy as np
from abc import ABC, abstractmethod
from dataclasses import dataclass
from typing import Dict, List, Any, Optional
import pickle
from pathlib import Path
import requests
import json
import time
from datetime import datetime

## Base Agent Class

All agents inherit from this base class, providing a common interface for execution and communication.

In [ ]:
@dataclass
class AgentMessage:
    """Message passed between agents"""
    sender: str
    recipient: str
    data: Any
    metadata: Dict[str, Any] = None
    
    def __post_init__(self):
        if self.metadata is None:
            self.metadata = {}


class BaseAgent(ABC):
    """Base class for all agents"""
    
    def __init__(self, name: str, config: Dict[str, Any] = None):
        self.name = name
        self.config = config or {}
        self.state = {}
        print(f"[{self.name}] Initialized")
    
    @abstractmethod
    def execute(self, message: AgentMessage) -> AgentMessage:
        """Execute the agent's primary task"""
        pass
    
    def log(self, message: str):
        """Log agent activity"""
        print(f"[{self.name}] {message}")
    
    def validate_input(self, message: AgentMessage, required_keys: List[str]):
        """Validate that message contains required data keys"""
        if not isinstance(message.data, dict):
            raise ValueError(f"Message data must be a dictionary")
        
        missing_keys = [key for key in required_keys if key not in message.data]
        if missing_keys:
            raise ValueError(f"Missing required keys: {missing_keys}")

## 1. DataLoaderAgent

Responsible for loading, filtering, and preprocessing complaint data.

In [ ]:
class DataLoaderAgent(BaseAgent):
    """Agent responsible for fetching and preprocessing complaint data from CFPB API"""
    
    def __init__(self, config: Dict[str, Any] = None):
        super().__init__("DataLoaderAgent", config)
        self.api_url = config.get('api_url', 'https://www.consumerfinance.gov/data-research/consumer-complaints/search/api/v1/')
        self.target_companies = config.get('target_companies', [
            'CAPITAL ONE FINANCIAL CORPORATION',
            'JPMORGAN CHASE & CO.',
            'Block, Inc.',
            'WELLS FARGO & COMPANY',
            'BANK OF AMERICA, NATIONAL ASSOCIATION',
            'CITIBANK, N.A.',
            'Early Warning Services, LLC',
            'NAVY FEDERAL CREDIT UNION',
            'SYNCHRONY FINANCIAL',
            'AMERICAN EXPRESS COMPANY',
            'DISCOVER BANK',
            'Paypal Holdings, Inc',
            'U.S. BANCORP',
            'Chime Financial Inc',
            'ALLY FINANCIAL INC.'
        ])
        self.date_cutoff = config.get('date_cutoff', '2024-11-01')
        self.max_narrative_length = config.get('max_narrative_length', 4930)
        self.rare_product_threshold = config.get('rare_product_threshold', 0.02)
        self.page_size = config.get('page_size', 100)
        self.max_records = config.get('max_records', 50000)
        self.use_api = config.get('use_api', True)
    
    def _fetch_complaints_from_api(self, company: str) -> List[Dict]:
        """Fetch complaints for a specific company from CFPB API"""
        all_complaints = []
        total_fetched = 0
        page = 1
        
        params = {
            'size': self.page_size,
            'company': company,
            'sort': 'created_date_desc',
            'date_received_min': self.date_cutoff,
            'format': 'json'
        }
        
        self.log(f"Fetching complaints for {company} from CFPB API...")
        
        while total_fetched < self.max_records:
            params['page'] = page
            
            try:
                response = requests.get(self.api_url, params=params, timeout=30)
                response.raise_for_status()
                
                data = response.json()
                
                # Handle different response formats
                if isinstance(data, dict):
                    complaints = data.get('results', [])
                elif isinstance(data, list):
                    complaints = data
                else:
                    complaints = []
                
                if not complaints:
                    break
                
                all_complaints.extend(complaints)
                total_fetched += len(complaints)
                
                self.log(f"  Page {page}: fetched {len(complaints)} complaints (total: {total_fetched})")
                
                page += 1
                time.sleep(0.5)  # Rate limiting
                
            except requests.exceptions.RequestException as e:
                self.log(f"  Error fetching page {page}: {e}")
                break
            except json.JSONDecodeError as e:
                self.log(f"  Error parsing JSON on page {page}: {e}")
                break
        
        self.log(f"Completed fetching {len(all_complaints)} complaints for {company}")
        return all_complaints[:self.max_records]
    
    def _load_from_csv(self, file_path: str) -> pd.DataFrame:
        """Load complaints from CSV file (fallback method)"""
        self.log(f"Loading data from CSV: {file_path}")
        dta = pd.read_csv(file_path)
        dta.columns = dta.columns.str.lower().str.replace(' ', '_')
        return dta
    
    def _preprocess_dataframe(self, dta: pd.DataFrame) -> pd.DataFrame:
        """Apply preprocessing filters to complaint data"""
        # Convert date
        if 'date_received' in dta.columns:
            dta['date_received'] = pd.to_datetime(dta['date_received']).dt.date
        
        # Filter by date
        initial_count = len(dta)
        dta = dta[dta['date_received'] >= pd.to_datetime(self.date_cutoff).date()]
        self.log(f"After date filter: {len(dta)} complaints (removed {initial_count - len(dta)})")
        
        # Filter by company
        if self.target_companies:
            initial_count = len(dta)
            dta = dta[dta.company.isin(self.target_companies)]
            self.log(f"After company filter: {len(dta)} complaints (removed {initial_count - len(dta)})")
        
        # Filter for non-null narratives
        initial_count = len(dta)
        w_narr = dta[dta.consumer_complaint_narrative.notnull()].copy()
        self.log(f"With narratives: {len(w_narr)} complaints (removed {initial_count - len(w_narr)})")
        
        # Filter by narrative length
        w_narr['narr_length'] = w_narr.consumer_complaint_narrative.str.len()
        initial_count = len(w_narr)
        w_narr = w_narr[w_narr.narr_length <= self.max_narrative_length]
        self.log(f"After length filter: {len(w_narr)} complaints (removed {initial_count - len(w_narr)})")
        
        # Remove rare products
        if 'product' in w_narr.columns and 'sub-product' in w_narr.columns:
            product_counts = w_narr[['product', 'sub-product']].value_counts()
            rare_products = product_counts[product_counts <= product_counts.sum() * self.rare_product_threshold].index.tolist()
            
            initial_count = len(w_narr)
            for product, subproduct in rare_products:
                w_narr = w_narr[~((w_narr.product == product) & (w_narr['sub-product'] == subproduct))]
            
            self.log(f"After rare product filter: {len(w_narr)} complaints (removed {initial_count - len(w_narr)})")
        
        return w_narr
    
    def execute(self, message: AgentMessage) -> AgentMessage:
        """Load and preprocess complaint data"""
        # Check if using API or CSV
        use_api = message.data.get('use_api', self.use_api)
        
        if use_api:
            # Fetch from CFPB API
            self.log("Fetching complaints from CFPB API...")
            all_complaints = []
            
            # Fetch for each company
            companies_to_fetch = message.data.get('companies', self.target_companies)
            
            for company in companies_to_fetch:
                company_complaints = self._fetch_complaints_from_api(company)
                all_complaints.extend(company_complaints)
            
            self.log(f"Total complaints fetched from API: {len(all_complaints)}")
            
            # Convert to DataFrame
            if all_complaints:
                dta = pd.DataFrame(all_complaints)
                # Normalize column names
                dta.columns = dta.columns.str.lower().str.replace(' ', '_')
            else:
                self.log("No complaints fetched from API")
                return AgentMessage(
                    sender=self.name,
                    recipient="EmbeddingAgent",
                    data={
                        'dataframe': pd.DataFrame(),
                        'n_complaints': 0,
                        'products': []
                    },
                    metadata={'status': 'no_data'}
                )
        else:
            # Load from CSV
            if 'file_path' not in message.data:
                raise ValueError("file_path required when not using API")
            
            file_path = message.data['file_path']
            dta = self._load_from_csv(file_path)
        
        self.log(f"Initial dataset: {len(dta)} complaints")
        
        # Preprocess
        w_narr = self._preprocess_dataframe(dta)
        
        # Store state
        self.state['data'] = w_narr
        
        return AgentMessage(
            sender=self.name,
            recipient="EmbeddingAgent",
            data={
                'dataframe': w_narr,
                'n_complaints': len(w_narr),
                'products': w_narr['product'].unique().tolist() if 'product' in w_narr.columns else []
            },
            metadata={'status': 'success'}
        )

## 2. EmbeddingAgent

Generates embeddings for complaint narratives using transformer models.

In [ ]:
class EmbeddingAgent(BaseAgent):
    """Agent responsible for generating text embeddings"""
    
    def __init__(self, config: Dict[str, Any] = None):
        super().__init__("EmbeddingAgent", config)
        self.model_name = config.get('model_name', 'sentence-transformers/all-miniLM-L6-v2')
        self.batch_size = config.get('batch_size', 32)
        self.output_path = config.get('output_path', '../data/complaint_embeddings.pkl')
        self.model = None
        self.tokenizer = None
        self.device = None
    
    def _initialize_model(self):
        """Lazy load the model when needed"""
        if self.model is None:
            self.log("Loading transformer model...")
            import torch
            from transformers import AutoTokenizer, AutoModel
            
            self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
            self.log(f"Using device: {self.device}")
            
            self.tokenizer = AutoTokenizer.from_pretrained(self.model_name)
            self.model = AutoModel.from_pretrained(self.model_name).to(self.device)
            self.log("Model loaded successfully")
    
    def _embed_text(self, texts: List[str]) -> np.ndarray:
        """Generate embeddings for a batch of texts"""
        import torch
        
        inputs = self.tokenizer(texts, return_tensors='pt', padding=True, truncation=True).to(self.device)
        
        with torch.no_grad():
            outputs = self.model(**inputs)
        
        embeddings = outputs.last_hidden_state.mean(dim=1)
        return embeddings.cpu().numpy()
    
    def execute(self, message: AgentMessage) -> AgentMessage:
        """Generate embeddings for all narratives"""
        self.validate_input(message, ['dataframe'])
        
        df = message.data['dataframe']
        self.log(f"Generating embeddings for {len(df)} complaints")
        
        # Initialize model
        self._initialize_model()
        
        # Generate embeddings in batches
        embeddings = []
        narratives = df['consumer_complaint_narrative'].tolist()
        
        for i in range(0, len(narratives), self.batch_size):
            batch = narratives[i:i + self.batch_size]
            batch_embeddings = self._embed_text(batch)
            embeddings.append(batch_embeddings)
            
            if (i // self.batch_size + 1) % 10 == 0:
                self.log(f"Processed batch {i // self.batch_size + 1}/{len(narratives) // self.batch_size + 1}")
        
        embeddings = np.vstack(embeddings)
        self.log(f"Generated embeddings with shape {embeddings.shape}")
        
        # Prepare output
        embeddings_dict = {
            'complaint_id': df['complaint_id'].tolist(),
            'product': df['product'].tolist(),
            'sub-product': df['sub-product'].tolist(),
            'company': df['company'].tolist(),
            'narrative': narratives,
            'embeddings': embeddings
        }
        
        # Save to file
        Path(self.output_path).parent.mkdir(parents=True, exist_ok=True)
        with open(self.output_path, 'wb') as f:
            pickle.dump(embeddings_dict, f)
        self.log(f"Saved embeddings to {self.output_path}")
        
        return AgentMessage(
            sender=self.name,
            recipient="ClusteringAgent",
            data={
                'embeddings_dict': embeddings_dict,
                'embeddings_path': self.output_path,
                'embedding_dim': embeddings.shape[1]
            },
            metadata={'status': 'success'}
        )

## 3. ClusteringAgent

Performs optimal clustering on embeddings using agglomerative clustering.

In [ ]:
class ClusteringAgent(BaseAgent):
    """Agent responsible for clustering embeddings"""
    
    def __init__(self, config: Dict[str, Any] = None):
        super().__init__("ClusteringAgent", config)
        self.sample_size = config.get('sample_size', 25000)
        self.thresholds = config.get('thresholds', [0.75, 1, 2.0, 3.0, 4.0, 4.5, 5.0, 5.5, 6.0, 10, 15, 20, 25, 28, 30, 35, 40, 45, 50])
        self.best_thresholds_path = config.get('best_thresholds_path', 'output/best_distance_thresholds.csv')
        self.output_path = config.get('output_path', '../data/complaint_embeddings_with_clusters.pkl')
    
    def _find_optimal_threshold(self, embeddings: np.ndarray, product: str) -> Dict[str, Any]:
        """Find optimal clustering threshold for a product"""
        from sklearn.cluster import AgglomerativeClustering
        from sklearn.metrics import silhouette_score
        from sklearn.utils import resample
        
        # Sample if needed
        if len(embeddings) > self.sample_size:
            embeddings_subset = resample(embeddings, n_samples=self.sample_size, random_state=442)
        else:
            embeddings_subset = embeddings
        
        self.log(f"Finding optimal threshold for {product} ({len(embeddings_subset)} samples)")
        
        best_score = -1
        best_result = None
        
        for thresh in self.thresholds:
            clustering = AgglomerativeClustering(n_clusters=None, distance_threshold=thresh)
            cluster_labels = clustering.fit_predict(embeddings_subset)
            n_clusters = len(set(cluster_labels))
            
            if n_clusters > 1 and n_clusters < len(embeddings_subset):
                sil_score = silhouette_score(embeddings_subset, cluster_labels)
                
                if sil_score > best_score:
                    best_score = sil_score
                    best_result = {
                        'product': product,
                        'best_distance_threshold': thresh,
                        'num_clusters': n_clusters,
                        'silhouette_score': sil_score
                    }
        
        self.log(f"Best threshold for {product}: {best_result['best_distance_threshold']} (clusters: {best_result['num_clusters']}, score: {best_result['silhouette_score']:.4f})")
        return best_result
    
    def _cluster_product(self, embeddings_df: pd.DataFrame, product: str, n_clusters: int, n_embedding_cols: int) -> pd.DataFrame:
        """Cluster embeddings for a specific product"""
        from sklearn.cluster import AgglomerativeClustering
        from sklearn.neighbors import NearestNeighbors
        
        product_mask = embeddings_df['product'] == product
        product_embeddings = embeddings_df.loc[product_mask].iloc[:, :n_embedding_cols].values
        
        self.log(f"Clustering {product}: {len(product_embeddings)} complaints into {n_clusters} clusters")
        
        if len(product_embeddings) > self.sample_size:
            # Sample and cluster
            sampled_indices = embeddings_df.loc[product_mask].sample(self.sample_size, random_state=442).index
            product_embeddings_sample = embeddings_df.loc[sampled_indices].iloc[:, :n_embedding_cols].values
            
            clustering = AgglomerativeClustering(n_clusters=n_clusters)
            cluster_labels_sample = clustering.fit_predict(product_embeddings_sample)
            
            embeddings_df.loc[sampled_indices, 'agglomerative_cluster'] = cluster_labels_sample
            
            # Assign remaining points
            remaining_mask = product_mask & ~embeddings_df.index.isin(sampled_indices)
            if remaining_mask.sum() > 0:
                remaining_embeddings = embeddings_df.loc[remaining_mask].iloc[:, :n_embedding_cols].values
                nn = NearestNeighbors(n_neighbors=1)
                nn.fit(product_embeddings_sample)
                distances, indices = nn.kneighbors(remaining_embeddings)
                remaining_cluster_labels = cluster_labels_sample[indices.flatten()]
                embeddings_df.loc[remaining_mask, 'agglomerative_cluster'] = remaining_cluster_labels
        else:
            # Cluster all points
            clustering = AgglomerativeClustering(n_clusters=n_clusters)
            cluster_labels = clustering.fit_predict(product_embeddings)
            embeddings_df.loc[product_mask, 'agglomerative_cluster'] = cluster_labels
        
        return embeddings_df
    
    def execute(self, message: AgentMessage) -> AgentMessage:
        """Perform clustering on embeddings"""
        self.validate_input(message, ['embeddings_dict'])
        
        embeddings_dict = message.data['embeddings_dict']
        
        # Convert to DataFrame
        embeddings_array = embeddings_dict['embeddings']
        embeddings_df = pd.DataFrame(embeddings_array)
        embeddings_df['complaint_id'] = embeddings_dict['complaint_id']
        embeddings_df['product'] = embeddings_dict['product']
        embeddings_df['sub-product'] = embeddings_dict['sub-product']
        embeddings_df['company'] = embeddings_dict['company']
        embeddings_df['narrative'] = embeddings_dict['narrative']
        
        n_embedding_cols = embeddings_array.shape[1]
        
        # Find optimal thresholds per product
        self.log("Finding optimal clustering thresholds per product...")
        best_thresholds = []
        
        for product in embeddings_df['product'].unique():
            product_mask = embeddings_df['product'] == product
            product_embeddings = embeddings_df.loc[product_mask].iloc[:, :n_embedding_cols].values
            
            result = self._find_optimal_threshold(product_embeddings, product)
            best_thresholds.append(result)
        
        best_thresh_df = pd.DataFrame(best_thresholds)
        Path(self.best_thresholds_path).parent.mkdir(parents=True, exist_ok=True)
        best_thresh_df.to_csv(self.best_thresholds_path, index=False)
        self.log(f"Saved best thresholds to {self.best_thresholds_path}")
        
        # Cluster each product
        self.log("Clustering all products...")
        for _, row in best_thresh_df.iterrows():
            embeddings_df = self._cluster_product(
                embeddings_df,
                row['product'],
                int(row['num_clusters']),
                n_embedding_cols
            )
        
        # Save clustered embeddings
        Path(self.output_path).parent.mkdir(parents=True, exist_ok=True)
        embeddings_df.to_pickle(self.output_path)
        self.log(f"Saved clustered embeddings to {self.output_path}")
        
        return AgentMessage(
            sender=self.name,
            recipient="NamingAgent",
            data={
                'embeddings_df': embeddings_df,
                'best_thresholds': best_thresh_df,
                'output_path': self.output_path
            },
            metadata={'status': 'success'}
        )

## 4. NamingAgent

Generates human-readable names for each cluster.

In [ ]:
class NamingAgent(BaseAgent):
    """Agent responsible for generating cluster names"""
    
    def __init__(self, config: Dict[str, Any] = None):
        super().__init__("NamingAgent", config)
        self.n_samples = config.get('n_samples', 10)
        self.output_path = config.get('output_path', 'output/cluster_names.csv')
    
    def _sample_narratives(self, embeddings_df: pd.DataFrame, product: str, cluster: int) -> List[str]:
        """Sample narratives from a specific cluster"""
        mask = (embeddings_df['product'] == product) & (embeddings_df['agglomerative_cluster'] == cluster)
        cluster_narratives = embeddings_df.loc[mask, 'narrative'].tolist()
        
        if len(cluster_narratives) <= self.n_samples:
            return cluster_narratives
        
        import random
        random.seed(442)
        return random.sample(cluster_narratives, self.n_samples)
    
    def _create_cluster_name(self, narratives: List[str]) -> str:
        """Generate a descriptive name for a cluster based on sample narratives"""
        # Import from name_clusters module if it exists, otherwise use simple heuristic
        try:
            from name_clusters import create_cluster_name
            return create_cluster_name(narratives)
        except ImportError:
            # Simple fallback: use most common keywords
            self.log("name_clusters module not found, using simple keyword extraction")
            from collections import Counter
            import re
            
            # Extract keywords
            all_words = []
            for narrative in narratives:
                words = re.findall(r'\b[a-zA-Z]{4,}\b', narrative.lower())
                all_words.extend(words)
            
            # Get most common words
            common_words = Counter(all_words).most_common(5)
            keywords = [word for word, _ in common_words if word not in ['that', 'with', 'this', 'from', 'have', 'been', 'were']]
            
            return ' '.join(keywords[:3]) if keywords else 'General Issues'
    
    def execute(self, message: AgentMessage) -> AgentMessage:
        """Generate names for all clusters"""
        self.validate_input(message, ['embeddings_df'])
        
        embeddings_df = message.data['embeddings_df']
        
        self.log("Generating cluster names...")
        cluster_names = []
        
        product_cluster_pairs = embeddings_df[['product', 'agglomerative_cluster']].drop_duplicates()
        
        for idx, (product, cluster) in enumerate(product_cluster_pairs.itertuples(index=False)):
            self.log(f"Processing cluster {idx + 1}/{len(product_cluster_pairs)}: {product} - Cluster {cluster}")
            
            sample_narratives = self._sample_narratives(embeddings_df, product, cluster)
            cluster_name = self._create_cluster_name(sample_narratives)
            
            cluster_names.append({
                'product': product,
                'agglomerative_cluster': cluster,
                'cluster_name': cluster_name
            })
        
        cluster_names_df = pd.DataFrame(cluster_names)
        
        # Save to file
        Path(self.output_path).parent.mkdir(parents=True, exist_ok=True)
        cluster_names_df.to_csv(self.output_path, index=False)
        self.log(f"Saved cluster names to {self.output_path}")
        
        # Merge names back to embeddings
        embeddings_with_names = embeddings_df.merge(
            cluster_names_df,
            on=['product', 'agglomerative_cluster'],
            how='left'
        )
        
        return AgentMessage(
            sender=self.name,
            recipient="VisualizationAgent",
            data={
                'embeddings_df': embeddings_with_names,
                'cluster_names_df': cluster_names_df
            },
            metadata={'status': 'success'}
        )

## 5. VisualizationAgent

Creates UMAP visualizations of clusters.

In [ ]:
class VisualizationAgent(BaseAgent):
    """Agent responsible for creating visualizations"""
    
    def __init__(self, config: Dict[str, Any] = None):
        super().__init__("VisualizationAgent", config)
        self.output_dir = config.get('output_dir', 'plots')
        self.n_neighbors = config.get('n_neighbors', 15)
        self.min_dist = config.get('min_dist', 0.1)
        self.dpi = config.get('dpi', 150)
    
    def _create_umap_plot(self, product: str, embeddings: np.ndarray, cluster_labels: np.ndarray, cluster_names: np.ndarray):
        """Create UMAP visualization for a product"""
        import umap
        import matplotlib.pyplot as plt
        from matplotlib.patches import Patch
        
        self.log(f"Creating UMAP visualization for {product}")
        
        # Reduce dimensions
        reducer = umap.UMAP(
            n_neighbors=self.n_neighbors,
            min_dist=self.min_dist,
            metric='cosine',
            random_state=42,
            init='random'
        )
        embedding_2d = reducer.fit_transform(embeddings)
        
        # Create plot
        plt.figure(figsize=(12, 8))
        scatter = plt.scatter(
            embedding_2d[:, 0],
            embedding_2d[:, 1],
            c=cluster_labels,
            cmap='Spectral',
            s=5,
            alpha=0.6
        )
        
        # Create legend
        unique_clusters = np.unique(cluster_labels)
        cluster_to_name = {}
        for cluster in unique_clusters:
            idx = np.where(cluster_labels == cluster)[0][0]
            cluster_to_name[cluster] = cluster_names[idx]
        
        legend_elements = []
        for i, cluster in enumerate(sorted(unique_clusters)[:50]):
            color = plt.cm.Spectral(i / max(len(unique_clusters) - 1, 1))
            cluster_name = cluster_to_name.get(cluster, "Unnamed")
            if pd.isna(cluster_name):
                cluster_name = "Unnamed"
            cluster_name_str = str(cluster_name)[:40]
            legend_elements.append(Patch(facecolor=color, label=f"{cluster}: {cluster_name_str}"))
        
        if len(unique_clusters) > 50:
            legend_elements.append(Patch(facecolor='white', label=f"... and {len(unique_clusters) - 50} more clusters"))
        
        plt.legend(handles=legend_elements, loc='center left', bbox_to_anchor=(1, 0.5), fontsize=8)
        plt.title(f'UMAP projection of {product} Embeddings ({len(unique_clusters)} clusters)')
        plt.xlabel('UMAP 1')
        plt.ylabel('UMAP 2')
        plt.tight_layout()
        
        # Save plot
        output_path = Path(self.output_dir) / f"umap_{product.replace('/', '_').replace(' ', '_')}.png"
        output_path.parent.mkdir(parents=True, exist_ok=True)
        plt.savefig(output_path, dpi=self.dpi, bbox_inches='tight')
        plt.close()
        
        self.log(f"Saved visualization to {output_path}")
        return str(output_path)
    
    def execute(self, message: AgentMessage) -> AgentMessage:
        """Create visualizations for all products"""
        self.validate_input(message, ['embeddings_df'])
        
        embeddings_df = message.data['embeddings_df']
        
        self.log("Creating UMAP visualizations...")
        
        # Determine embedding columns
        non_embedding_cols = ['complaint_id', 'product', 'sub-product', 'company', 'narrative', 'agglomerative_cluster', 'cluster_name']
        embedding_cols = [col for col in embeddings_df.columns if col not in non_embedding_cols]
        
        visualization_paths = []
        
        for product in embeddings_df['product'].unique():
            product_mask = embeddings_df['product'] == product
            product_embeddings = embeddings_df.loc[product_mask, embedding_cols].values
            cluster_labels = embeddings_df.loc[product_mask, 'agglomerative_cluster'].values
            cluster_names = embeddings_df.loc[product_mask, 'cluster_name'].values
            
            viz_path = self._create_umap_plot(product, product_embeddings, cluster_labels, cluster_names)
            visualization_paths.append(viz_path)
        
        self.log(f"Created {len(visualization_paths)} visualizations")
        
        return AgentMessage(
            sender=self.name,
            recipient="CoordinatorAgent",
            data={
                'visualization_paths': visualization_paths,
                'embeddings_df': embeddings_df
            },
            metadata={'status': 'success'}
        )

## 6. CoordinatorAgent

Orchestrates the entire pipeline by coordinating all agents.

In [ ]:
class CoordinatorAgent(BaseAgent):
    """Agent responsible for coordinating the entire pipeline"""
    
    def __init__(self, config: Dict[str, Any] = None):
        super().__init__("CoordinatorAgent", config)
        self.agents = {}
    
    def register_agent(self, agent: BaseAgent):
        """Register an agent with the coordinator"""
        self.agents[agent.name] = agent
        self.log(f"Registered agent: {agent.name}")
    
    def execute(self, message: AgentMessage) -> AgentMessage:
        """Execute the full pipeline"""
        self.log("Starting multi-agent complaint analysis pipeline")
        self.log("="*60)
        
        # 1. Data Loading
        self.log("\nPhase 1: Data Loading")
        self.log("-"*60)
        data_loader = self.agents.get('DataLoaderAgent')
        if not data_loader:
            raise ValueError("DataLoaderAgent not registered")
        
        loader_result = data_loader.execute(message)
        
        # 2. Embedding Generation
        self.log("\nPhase 2: Embedding Generation")
        self.log("-"*60)
        embedding_agent = self.agents.get('EmbeddingAgent')
        if not embedding_agent:
            raise ValueError("EmbeddingAgent not registered")
        
        embedding_result = embedding_agent.execute(loader_result)
        
        # 3. Clustering
        self.log("\nPhase 3: Clustering")
        self.log("-"*60)
        clustering_agent = self.agents.get('ClusteringAgent')
        if not clustering_agent:
            raise ValueError("ClusteringAgent not registered")
        
        clustering_result = clustering_agent.execute(embedding_result)
        
        # 4. Naming
        self.log("\nPhase 4: Cluster Naming")
        self.log("-"*60)
        naming_agent = self.agents.get('NamingAgent')
        if not naming_agent:
            raise ValueError("NamingAgent not registered")
        
        naming_result = naming_agent.execute(clustering_result)
        
        # 5. Visualization
        self.log("\nPhase 5: Visualization")
        self.log("-"*60)
        viz_agent = self.agents.get('VisualizationAgent')
        if not viz_agent:
            raise ValueError("VisualizationAgent not registered")
        
        viz_result = viz_agent.execute(naming_result)
        
        # Summary
        self.log("\n" + "="*60)
        self.log("Pipeline completed successfully!")
        self.log("="*60)
        
        return viz_result

## Pipeline Execution

Now let's set up and run the complete multi-agent pipeline.

In [ ]:
# Configuration
config = {
    'data_loader': {
        'api_url': 'https://www.consumerfinance.gov/data-research/consumer-complaints/search/api/v1/',
        'use_api': True,  # Set to False to use CSV file instead
        'target_companies': [
            'CAPITAL ONE FINANCIAL CORPORATION',
            'JPMORGAN CHASE & CO.',
            'Block, Inc.',
            'WELLS FARGO & COMPANY',
            'BANK OF AMERICA, NATIONAL ASSOCIATION',
            'CITIBANK, N.A.',
            'Early Warning Services, LLC',
            'NAVY FEDERAL CREDIT UNION',
            'SYNCHRONY FINANCIAL',
            'AMERICAN EXPRESS COMPANY',
            'DISCOVER BANK',
            'Paypal Holdings, Inc',
            'U.S. BANCORP',
            'Chime Financial Inc',
            'ALLY FINANCIAL INC.'
        ],
        'date_cutoff': '2024-11-01',
        'max_narrative_length': 4930,
        'rare_product_threshold': 0.02,
        'page_size': 100,
        'max_records': 50000
    },
    'embedding': {
        'model_name': 'sentence-transformers/all-miniLM-L6-v2',
        'batch_size': 32,
        'output_path': '../data/complaint_embeddings.pkl'
    },
    'clustering': {
        'sample_size': 25000,
        'best_thresholds_path': 'output/best_distance_thresholds.csv',
        'output_path': '../data/complaint_embeddings_with_clusters.pkl'
    },
    'naming': {
        'n_samples': 10,
        'output_path': 'output/cluster_names.csv'
    },
    'visualization': {
        'output_dir': 'plots',
        'n_neighbors': 15,
        'min_dist': 0.1,
        'dpi': 150
    }
}

print("Configuration loaded")
print(f"Data source: {'CFPB API' if config['data_loader']['use_api'] else 'CSV file'}")

In [ ]:
# Initialize all agents
data_loader = DataLoaderAgent(config['data_loader'])
embedding_agent = EmbeddingAgent(config['embedding'])
clustering_agent = ClusteringAgent(config['clustering'])
naming_agent = NamingAgent(config['naming'])
viz_agent = VisualizationAgent(config['visualization'])
coordinator = CoordinatorAgent()

# Register agents with coordinator
coordinator.register_agent(data_loader)
coordinator.register_agent(embedding_agent)
coordinator.register_agent(clustering_agent)
coordinator.register_agent(naming_agent)
coordinator.register_agent(viz_agent)

print("\nAll agents initialized and registered")

In [ ]:
# Create initial message and run pipeline
# For API mode: specify use_api=True and optionally companies list
# For CSV mode: specify use_api=False and file_path

if config['data_loader']['use_api']:
    # Fetch from CFPB API
    initial_message = AgentMessage(
        sender="User",
        recipient="CoordinatorAgent",
        data={
            'use_api': True,
            'companies': config['data_loader']['target_companies'][:3]  # Limit to 3 companies for demo
        },
        metadata={'timestamp': pd.Timestamp.now().isoformat()}
    )
else:
    # Load from CSV file
    initial_message = AgentMessage(
        sender="User",
        recipient="CoordinatorAgent",
        data={
            'use_api': False,
            'file_path': '../data/complaints.csv'
        },
        metadata={'timestamp': pd.Timestamp.now().isoformat()}
    )

print(f"Starting pipeline with {'API' if config['data_loader']['use_api'] else 'CSV'} data source...")

# Execute pipeline
result = coordinator.execute(initial_message)

## Results Analysis

Let's examine the results from the pipeline.

In [ ]:
# Load and display results
embeddings_df = result.data['embeddings_df']

print(f"Total complaints processed: {len(embeddings_df)}")
print(f"\nProducts analyzed: {embeddings_df['product'].nunique()}")
print(f"Total clusters created: {embeddings_df['agglomerative_cluster'].nunique()}")
print(f"\nVisualizations created: {len(result.data['visualization_paths'])}")

# Display cluster distribution
print("\nCluster distribution by product:")
cluster_summary = embeddings_df.groupby('product')['agglomerative_cluster'].nunique().sort_values(ascending=False)
print(cluster_summary)

In [ ]:
# Display sample cluster names
print("Sample cluster names:")
sample_clusters = embeddings_df[['product', 'agglomerative_cluster', 'cluster_name']].drop_duplicates().head(10)
print(sample_clusters.to_string(index=False))

In [ ]:
# Display visualization paths
print("Visualization files created:")
for path in result.data['visualization_paths']:
    print(f"  - {path}")

## Agent Communication Log

The multi-agent framework provides transparency through agent communication logs. Each agent logs its activities, making the pipeline easy to debug and monitor.

In [ ]:
# Display agent states (optional)
print("Agent States:")
print(f"DataLoaderAgent: {list(coordinator.agents['DataLoaderAgent'].state.keys())}")
print(f"EmbeddingAgent: Model loaded = {coordinator.agents['EmbeddingAgent'].model is not None}")
print(f"ClusteringAgent: {list(coordinator.agents['ClusteringAgent'].state.keys())}")
print(f"NamingAgent: {list(coordinator.agents['NamingAgent'].state.keys())}")
print(f"VisualizationAgent: {list(coordinator.agents['VisualizationAgent'].state.keys())}")